<a href="https://colab.research.google.com/github/ThayseBachmeyer/ProcessamentoSinaisI/blob/main/Aula%2007/C%C3%B3digos/prat7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Código utilizado na questão 1

In [ ]:
import os, json
import numpy as np
import scipy.signal as sig
from scipy.io import wavfile
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Image, display
import glob

OUT = "figs"
os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({"font.size": 6.5, "axes.titlesize": 7, "axes.labelsize": 6.5,
"xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 5.5,
"xtick.major.size": 2, "ytick.major.size": 2, "lines.linewidth": 0.9,
"axes.grid": True, "grid.alpha": .35,
"figure.dpi": 220, "savefig.dpi": 220, "savefig.bbox": "tight"})
RES = {}

def block(tipo, fc, fs, r):
  th = 2 * np.pi * fc / fs
  a = np.array([1, -2 * r * np.cos(th), r ** 2])
  if tipo == "lp":
    b = np.array([1, 2, 1]); w0 = 0
  elif tipo == "hp":
    b = np.array([1, -2, 1]); w0 = np.pi
  elif tipo == "bp":
    b = np.array([1, 0, -1]); w0 = th
  elif tipo == "notch":
    b = np.array([1, -2 * np.cos(th), 1]); w0 = 0
  _, h = sig.freqz(b, a, worN=[w0])
  return b / abs(h[0]), a

def cascade(blocks):
  b, a = np.array([1.0]), np.array([1.0])
  for bi, ai in blocks:
    b, a = np.convolve(b, bi), np.convolve(a, ai)
  return b, a

def parallel(blocks):

  b, a = np.array([0.0]), np.array([1.0])
  for bi, ai in blocks:
    b = np.polyadd(np.convolve(b, ai), np.convolve(bi, a))
    a = np.convolve(a, ai)
  return b, a

def freq(b, a, fs, n=4096):

  f, h = sig.freqz(b, a, worN=n, fs=fs)
  return f, 20 * np.log10(np.abs(h) + 1e-12)


def pz(ax, b, a, title="", leg=True):

  z, p = np.roots(b), np.roots(a)
  t = np.linspace(0, 2 * np.pi, 400)
  ax.plot(np.cos(t), np.sin(t), "k--", lw=.7)
  ax.plot(z.real, z.imag, "o", mfc="none", mec="b", ms=5, label="zeros")
  ax.plot(p.real, p.imag, "x", c="r", ms=5, label="polos")
  ax.set_aspect("equal"); ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
  ax.set_xlabel("Re{z}"); ax.set_ylabel("Im{z}"); ax.set_title(title)
  if leg: ax.legend(fontsize=5, loc="upper right", framealpha=.6)

def bandwidth3db(b, a, fs):

  f, H = freq(b, a, fs, 20000)
  m = H > H.max() - 3
  return f[m].min(), f[m].max()

def save(fig, name):
  fig.savefig(f"{OUT}/{name}.png"); plt.close(fig)

def quant(c, bits):

  c = np.asarray(c, float)
  ib = max(0, int(np.ceil(np.log2(np.max(np.abs(c)) + 1e-12))))
  fb = bits - 1 - ib
  return np.round(c * 2.0 ** fb) / 2.0 ** fb

def quantb(b, bits):

  b = np.asarray(b, float)
  i = np.flatnonzero(np.abs(b) > 1e-12)[0]
  g = b[i]
  return quant([g], bits)[0] * quant(b / g, bits)

def maxpole(a):

  p = np.roots(a)
  return float(np.max(np.abs(p))) if len(p) else 0.0

fs = 20000
RS = [0.5, 0.8, 0.9, 0.95, 0.99] # valores de r investigados
Q1 = {"a": ("lp", 1000, "Passa-baixas, fc = 1000 Hz"),
      "b": ("hp", 2000, "Passa-altas, fc = 2000 Hz"),
      "c": ("bp", 7000, "Passa-faixas, fc = 7000 Hz"),
      "d": ("notch", 3000, "Notch, fc = 3000 Hz")}
RES["q1"] = {}
for k, (tp, fc, tit) in Q1.items():
    fig, ax = plt.subplots(2, 1, figsize=(3.4, 5.0), gridspec_kw={"height_ratios": [1, 1.2]})
    RES["q1"][k] = {}
    for r in RS:
        b, a = block(tp, fc, fs, r)
        f, H = freq(b, a, fs)
        ax[0].plot(f, H, label=f"r = {r}")
        lo, hi = bandwidth3db(b, a, fs) if tp in ("bp",) else (None, None)
        RES["q1"][k][str(r)] = {"b": b.round(5).tolist(), "a": a.round(5).tolist(),
                                "bw": None if lo is None else round(hi - lo, 1)}
        if tp == "notch": # largura do entalhe (-3 dB)
            fN, HN = freq(b, a, fs, 20000); m = HN < -3
            RES["q1"][k][str(r)]["bw"] = round(fN[m].max() - fN[m].min(), 1)
    ax[0].axvline(fc, c="k", ls=":", lw=.8)
    ax[0].set_ylim(-60, 25 if tp in ("lp", "hp") else 5)
    ax[0].set_xlabel("Frequência (Hz)"); ax[0].set_ylabel("|H| (dB)")
    ax[0].set_title(tit); ax[0].legend(fontsize=5.5)
    b, a = block(tp, fc, fs, 0.9)
    pz(ax[1], b, a, "Polos e zeros (r = 0,9)")
    for r in [0.5, 0.99]: # polos para r extremos em cinza
        p = np.roots(block(tp, fc, fs, r)[1])
        ax[1].plot(p.real, p.imag, "x", c="gray", ms=5)
    fig.tight_layout();
    save(fig, f"q1{k}") # -> q1a.png, q1b.png, q1c.png, q1d.png

    display(Image(filename=f"{OUT}/q1{k}.png"))

##Código utilizado na questão 2

In [ ]:
import os, json
import numpy as np
import scipy.signal as sig
from scipy.io import wavfile
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
import glob

OUT = "figs"
os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({"font.size": 6.5, "axes.titlesize": 7, "axes.labelsize": 6.5,
"xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 5.5,
"xtick.major.size": 2, "ytick.major.size": 2, "lines.linewidth": 0.9,
"axes.grid": True, "grid.alpha": .35,
"figure.dpi": 220, "savefig.dpi": 220, "savefig.bbox": "tight"})
RES = {}

def block(tipo, fc, fs, r):
  th = 2 * np.pi * fc / fs
  a = np.array([1, -2 * r * np.cos(th), r ** 2])
  if tipo == "lp":
    b = np.array([1, 2, 1]); w0 = 0
  elif tipo == "hp":
    b = np.array([1, -2, 1]); w0 = np.pi
  elif tipo == "bp":
    b = np.array([1, 0, -1]); w0 = th
  elif tipo == "notch":
    b = np.array([1, -2 * np.cos(th), 1]); w0 = 0
  _, h = sig.freqz(b, a, worN=[w0])
  return b / abs(h[0]), a

def cascade(blocks):
  b, a = np.array([1.0]), np.array([1.0])
  for bi, ai in blocks:
    b, a = np.convolve(b, bi), np.convolve(a, ai)
  return b, a

def parallel(blocks):

  b, a = np.array([0.0]), np.array([1.0])
  for bi, ai in blocks:
    b = np.polyadd(np.convolve(b, ai), np.convolve(bi, a))
    a = np.convolve(a, ai)
  return b, a

def freq(b, a, fs, n=4096):

  f, h = sig.freqz(b, a, worN=n, fs=fs)
  return f, 20 * np.log10(np.abs(h) + 1e-12)


def pz(ax, b, a, title="", leg=True):

  z, p = np.roots(b), np.roots(a)
  t = np.linspace(0, 2 * np.pi, 400)
  ax.plot(np.cos(t), np.sin(t), "k--", lw=.7)
  ax.plot(z.real, z.imag, "o", mfc="none", mec="b", ms=5, label="zeros")
  ax.plot(p.real, p.imag, "x", c="r", ms=5, label="polos")
  ax.set_aspect("equal"); ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
  ax.set_xlabel("Re{z}"); ax.set_ylabel("Im{z}"); ax.set_title(title)
  if leg: ax.legend(fontsize=5, loc="upper right", framealpha=.6)

def bandwidth3db(b, a, fs):

  f, H = freq(b, a, fs, 20000)
  m = H > H.max() - 3
  return f[m].min(), f[m].max()

def save(fig, name):
  fig.savefig(f"{OUT}/{name}.png"); plt.close(fig)

def quant(c, bits):

  c = np.asarray(c, float)
  ib = max(0, int(np.ceil(np.log2(np.max(np.abs(c)) + 1e-12))))
  fb = bits - 1 - ib
  return np.round(c * 2.0 ** fb) / 2.0 ** fb

def quantb(b, bits):

  b = np.asarray(b, float)
  i = np.flatnonzero(np.abs(b) > 1e-12)[0]
  g = b[i]
  return quant([g], bits)[0] * quant(b / g, bits)

def maxpole(a):

  p = np.roots(a)
  return float(np.max(np.abs(p))) if len(p) else 0.0

r2 = 0.82
bp2 = [block("bp", 6450, fs, r2), block("bp", 8100, fs, r2)] # polos escalonados
b2, a2 = cascade(bp2)
_, h = sig.freqz(b2, a2, worN=[2 * np.pi * 7000 / fs]); b2 = b2 / abs(h[0]) # 0 dB em 7 kHz
bp2[0] = (bp2[0][0] / abs(h[0]), bp2[0][1])
b1c, a1c = block("bp", 7000, fs, 0.9) # filtro da Q1(c) para comparação
fig, ax = plt.subplots(2, 1, figsize=(3.4, 5.0), gridspec_kw={"height_ratios": [1, 1.2]})
f, H = freq(b2, a2, fs); ax[0].plot(f, H, label="Cascata Q2 (4ª ordem)")
f, H = freq(b1c, a1c, fs); ax[0].plot(f, H, "--", label="Q1(c), r = 0,9")
for x in (6000, 8000): ax[0].axvline(x, c="k", ls=":", lw=.8)
ax[0].axhline(-3, c="gray", ls=":", lw=.8)
ax[0].set_ylim(-70, 5); ax[0].set_xlabel("Frequência (Hz)"); ax[0].set_ylabel("|H| (dB)")
ax[0].set_title("Passa-faixas 6–8 kHz"); ax[0].legend(fontsize=5.5)
pz(ax[1], b2, a2, "Polos e zeros (Q2)")
fig.tight_layout(); save(fig, "q2") # -> q2.png
lo, hi = bandwidth3db(b2, a2, fs); lo1, hi1 = bandwidth3db(b1c, a1c, fs)
RES["q2"] = {"bw": [round(lo), round(hi)], "bw_q1": [round(lo1), round(hi1)],
              "b": b2.round(5).tolist(), "a": a2.round(5).tolist(), "r": r2}


display(Image(filename="figs/q2.png"))

##Código utilizado na questão 3

In [ ]:
import os, json
import numpy as np
import scipy.signal as sig
from scipy.io import wavfile
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Image, display
import glob

OUT = "figs"
os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({"font.size": 6.5, "axes.titlesize": 7, "axes.labelsize": 6.5,
"xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 5.5,
"xtick.major.size": 2, "ytick.major.size": 2, "lines.linewidth": 0.9,
"axes.grid": True, "grid.alpha": .35,
"figure.dpi": 220, "savefig.dpi": 220, "savefig.bbox": "tight"})
RES = {}

def block(tipo, fc, fs, r):
  th = 2 * np.pi * fc / fs
  a = np.array([1, -2 * r * np.cos(th), r ** 2])
  if tipo == "lp":
    b = np.array([1, 2, 1]); w0 = 0
  elif tipo == "hp":
    b = np.array([1, -2, 1]); w0 = np.pi
  elif tipo == "bp":
    b = np.array([1, 0, -1]); w0 = th
  elif tipo == "notch":
    b = np.array([1, -2 * np.cos(th), 1]); w0 = 0
  _, h = sig.freqz(b, a, worN=[w0])
  return b / abs(h[0]), a

def cascade(blocks):
  b, a = np.array([1.0]), np.array([1.0])
  for bi, ai in blocks:
    b, a = np.convolve(b, bi), np.convolve(a, ai)
  return b, a

def parallel(blocks):

  b, a = np.array([0.0]), np.array([1.0])
  for bi, ai in blocks:
    b = np.polyadd(np.convolve(b, ai), np.convolve(bi, a))
    a = np.convolve(a, ai)
  return b, a

def freq(b, a, fs, n=4096):

  f, h = sig.freqz(b, a, worN=n, fs=fs)
  return f, 20 * np.log10(np.abs(h) + 1e-12)


def pz(ax, b, a, title="", leg=True):

  z, p = np.roots(b), np.roots(a)
  t = np.linspace(0, 2 * np.pi, 400)
  ax.plot(np.cos(t), np.sin(t), "k--", lw=.7)
  ax.plot(z.real, z.imag, "o", mfc="none", mec="b", ms=5, label="zeros")
  ax.plot(p.real, p.imag, "x", c="r", ms=5, label="polos")
  ax.set_aspect("equal"); ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
  ax.set_xlabel("Re{z}"); ax.set_ylabel("Im{z}"); ax.set_title(title)
  if leg: ax.legend(fontsize=5, loc="upper right", framealpha=.6)

def bandwidth3db(b, a, fs):

  f, H = freq(b, a, fs, 20000)
  m = H > H.max() - 3
  return f[m].min(), f[m].max()

def save(fig, name):
  fig.savefig(f"{OUT}/{name}.png"); plt.close(fig)

def quant(c, bits):

  c = np.asarray(c, float)
  ib = max(0, int(np.ceil(np.log2(np.max(np.abs(c)) + 1e-12))))
  fb = bits - 1 - ib
  return np.round(c * 2.0 ** fb) / 2.0 ** fb

def quantb(b, bits):

  b = np.asarray(b, float)
  i = np.flatnonzero(np.abs(b) > 1e-12)[0]
  g = b[i]
  return quant([g], bits)[0] * quant(b / g, bits)

def maxpole(a):

  p = np.roots(a)
  return float(np.max(np.abs(p))) if len(p) else 0.0

r3, f3 = 0.58, 1500
one = (np.array([1.0]), np.array([1.0])) # ramo direto (ganho 1)
bpb, bpa = block("bp", f3, fs, r3)
lp3, hp3 = one, (-bpb, bpa) # H(z) = 1 - H_PF(z)
b3, a3 = parallel([lp3, hp3])
bn, an = block("notch", 3000, fs, 0.9) # notch da Q1(d) para comparação
fig, ax = plt.subplots(2, 1, figsize=(3.4, 5.0), gridspec_kw={"height_ratios": [1, 1.2]})
f, H = freq(b3, a3, fs); ax[0].plot(f, H, label="Paralelo 1 − H_PF(z)")
f, H = freq(bn, an, fs); ax[0].plot(f, H, "--", label="Notch Q1(d), r = 0,9")
for x in (1000, 4000): ax[0].axvline(x, c="k", ls=":", lw=.8)
ax[0].set_ylim(-60, 10); ax[0].set_xlabel("Frequência (Hz)"); ax[0].set_ylabel("|H| (dB)")
ax[0].set_title("Rejeita-faixas 1–4 kHz"); ax[0].legend(fontsize=5.5)
pz(ax[1], b3, a3, "Polos e zeros (Q3)")
fig.tight_layout(); save(fig, "q3") # -> q3.png
f, H = freq(b3, a3, fs, 20000)
RES["q3"] = {"min_db": round(float(H.min()), 1), "fmin": round(float(f[np.argmin(H)])),
             "b": b3.round(5).tolist(), "a": a3.round(5).tolist(), "r": r3,
             "H1k": round(float(np.interp(1000, f, H)), 1), "H4k": round(float(np.interp(4000, f, H)), 1),
             "bw3": [round(float(f[H < -3].min())), round(float(f[H < -3].max()))]}
display(Image(filename="figs/q3.png"))

##Código utilizado na questão 4

In [ ]:
import os, json
import numpy as np
import scipy.signal as sig
from scipy.io import wavfile
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Image, display
import glob

OUT = "figs"
os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({"font.size": 6.5, "axes.titlesize": 7, "axes.labelsize": 6.5,
"xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 5.5,
"xtick.major.size": 2, "ytick.major.size": 2, "lines.linewidth": 0.9,
"axes.grid": True, "grid.alpha": .35,
"figure.dpi": 220, "savefig.dpi": 220, "savefig.bbox": "tight"})
RES = {}

def block(tipo, fc, fs, r):
  th = 2 * np.pi * fc / fs
  a = np.array([1, -2 * r * np.cos(th), r ** 2])
  if tipo == "lp":
    b = np.array([1, 2, 1]); w0 = 0
  elif tipo == "hp":
    b = np.array([1, -2, 1]); w0 = np.pi
  elif tipo == "bp":
    b = np.array([1, 0, -1]); w0 = th
  elif tipo == "notch":
    b = np.array([1, -2 * np.cos(th), 1]); w0 = 0
  _, h = sig.freqz(b, a, worN=[w0])
  return b / abs(h[0]), a

def cascade(blocks):
  b, a = np.array([1.0]), np.array([1.0])
  for bi, ai in blocks:
    b, a = np.convolve(b, bi), np.convolve(a, ai)
  return b, a

def parallel(blocks):

  b, a = np.array([0.0]), np.array([1.0])
  for bi, ai in blocks:
    b = np.polyadd(np.convolve(b, ai), np.convolve(bi, a))
    a = np.convolve(a, ai)
  return b, a

def freq(b, a, fs, n=4096):

  f, h = sig.freqz(b, a, worN=n, fs=fs)
  return f, 20 * np.log10(np.abs(h) + 1e-12)


def pz(ax, b, a, title="", leg=True):

  z, p = np.roots(b), np.roots(a)
  t = np.linspace(0, 2 * np.pi, 400)
  ax.plot(np.cos(t), np.sin(t), "k--", lw=.7)
  ax.plot(z.real, z.imag, "o", mfc="none", mec="b", ms=5, label="zeros")
  ax.plot(p.real, p.imag, "x", c="r", ms=5, label="polos")
  ax.set_aspect("equal"); ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
  ax.set_xlabel("Re{z}"); ax.set_ylabel("Im{z}"); ax.set_title(title)
  if leg: ax.legend(fontsize=5, loc="upper right", framealpha=.6)

def bandwidth3db(b, a, fs):

  f, H = freq(b, a, fs, 20000)
  m = H > H.max() - 3
  return f[m].min(), f[m].max()

def save(fig, name):
  fig.savefig(f"{OUT}/{name}.png"); plt.close(fig)

def quant(c, bits):

  c = np.asarray(c, float)
  ib = max(0, int(np.ceil(np.log2(np.max(np.abs(c)) + 1e-12))))
  fb = bits - 1 - ib
  return np.round(c * 2.0 ** fb) / 2.0 ** fb

def quantb(b, bits):

  b = np.asarray(b, float)
  i = np.flatnonzero(np.abs(b) > 1e-12)[0]
  g = b[i]
  return quant([g], bits)[0] * quant(b / g, bits)

def maxpole(a):

  p = np.roots(a)
  return float(np.max(np.abs(p))) if len(p) else 0.0

BITS4 = [2, 4, 8, 16, 32]
RES["q4"] = {}
for nome, (bt, at), blocos, modo in [("q2", (b2, a2), bp2, "cascata"),
                                     ("q3", (b3, a3), [lp3, hp3], "paralelo")]:
    fig, ax = plt.subplots(2, 2, figsize=(3.5, 3.9))
    f, H = freq(bt, at, fs)
    ax[0, 0].plot(f, H, "k", lw=2, label="original"); ax[0, 1].plot(f, H, "k", lw=2, label="original")
    RES["q4"][nome] = {}
    for bits in BITS4:
        bq, aq = quantb(bt, bits), quant(at, bits) # (1) filtro resultante
        qb = [(quantb(b_, bits), quant(a_, bits)) for b_, a_ in blocos] # (2) cada bloco
        bb, ab = cascade(qb) if modo == "cascata" else parallel(qb)
        mp_d, mp_b = maxpole(aq), maxpole(ab)
        for col, (B, A) in enumerate([(bq, aq), (bb, ab)]):
            if np.allclose(B, 0):
                continue
            fq, Hq = freq(B, A, fs)
            ax[0, col].plot(fq, Hq, lw=.9, label=f"{bits} bits")
        RES["q4"][nome][bits] = {"maxpole_direta": round(mp_d, 4), "maxpole_blocos": round(mp_b, 4),
                                 "estavel_direta": bool(mp_d < 1), "estavel_blocos": bool(mp_b < 1)}
        if bits == 8: # diagramas para 8 bits
            pz(ax[1, 0], bq, aq, "Direta, 8 bits"); pz(ax[1, 1], bb, ab, "Por blocos, 8 bits")
            p0 = np.roots(at); ax[1, 0].plot(p0.real, p0.imag, "+", c="k", ms=7)
            ax[1, 1].plot(p0.real, p0.imag, "+", c="k", ms=7)
    for col, t in enumerate(["Filtro resultante", "Por bloco"]):
        ax[0, col].set_ylim(-70, 20); ax[0, col].set_title(t)
        ax[0, col].set_xlabel("Frequência (Hz)"); ax[0, col].set_ylabel("|H| (dB)")
        ax[0, col].legend(fontsize=5)
    fig.tight_layout(); save(fig, f"q4_{nome}") # -> q4_q2.png, q4_q3.png

    display(Image(filename=f"figs/q4_{nome}.png"))

##Código utilizado na questão 5-a

In [ ]:
import os, json, urllib.request
import numpy as np
import scipy.signal as sig
from scipy.io import wavfile
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Image, display

OUT = "figs"
os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({
    "font.size": 6.5, "axes.titlesize": 7, "axes.labelsize": 6.5,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 5.5,
    "xtick.major.size": 2, "ytick.major.size": 2, "lines.linewidth": 0.9,
    "axes.grid": True, "grid.alpha": .35,
    "figure.dpi": 220, "savefig.dpi": 220, "savefig.bbox": "tight"
})
RES = {}

def block(tipo, fc, fs, r):
    th = 2 * np.pi * fc / fs
    a = np.array([1, -2 * r * np.cos(th), r ** 2])
    if tipo == "lp":
        b = np.array([1, 2, 1]); w0 = 0
    elif tipo == "hp":
        b = np.array([1, -2, 1]); w0 = np.pi
    elif tipo == "bp":
        b = np.array([1, 0, -1]); w0 = th
    elif tipo == "notch":
        b = np.array([1, -2 * np.cos(th), 1]); w0 = 0
    _, h = sig.freqz(b, a, worN=[w0])
    return b / abs(h[0]), a

def cascade(blocks):
    b, a = np.array([1.0]), np.array([1.0])
    for bi, ai in blocks:
        b, a = np.convolve(b, bi), np.convolve(a, ai)
    return b, a

def parallel(blocks):
    b, a = np.array([0.0]), np.array([1.0])
    for bi, ai in blocks:
        b = np.polyadd(np.convolve(b, ai), np.convolve(bi, a))
        a = np.convolve(a, ai)
    return b, a

def freq(b, a, fs, n=4096):
    f, h = sig.freqz(b, a, worN=n, fs=fs)
    return f, 20 * np.log10(np.abs(h) + 1e-12)

def pz(ax, b, a, title="", leg=True):
    z, p = np.roots(b), np.roots(a)
    t = np.linspace(0, 2 * np.pi, 400)
    ax.plot(np.cos(t), np.sin(t), "k--", lw=.7)
    ax.plot(z.real, z.imag, "o", mfc="none", mec="b", ms=5, label="zeros")
    ax.plot(p.real, p.imag, "x", c="r", ms=5, label="polos")
    ax.set_aspect("equal"); ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
    ax.set_xlabel("Re{z}"); ax.set_ylabel("Im{z}"); ax.set_title(title)
    if leg: ax.legend(fontsize=5, loc="upper right", framealpha=.6)

def bandwidth3db(b, a, fs):
    f, H = freq(b, a, fs, 20000)
    m = H > H.max() - 3
    return f[m].min(), f[m].max()

def save(fig, name):
    fig.savefig(f"{OUT}/{name}.png"); plt.close(fig)

def quant(c, bits):
    c = np.asarray(c, float)
    ib = max(0, int(np.ceil(np.log2(np.max(np.abs(c)) + 1e-12))))
    fb = bits - 1 - ib
    return np.round(c * 2.0 ** fb) / 2.0 ** fb

def quantb(b, bits):
    b = np.asarray(b, float)
    i = np.flatnonzero(np.abs(b) > 1e-12)[0]
    g = b[i]
    return quant([g], bits)[0] * quant(b / g, bits)

def maxpole(a):
    p = np.roots(a)
    return float(np.max(np.abs(p))) if len(p) else 0.0

# Baixa automaticamente o handel.wav caso não esteja na pasta do Colab
if not os.path.exists("handel.wav"):
    try:
        url = "https://github.com/ThayseBachmeyer/ProcessamentoSinaisI/raw/refs/heads/main/Aula%2005/Dados/handel.wav"
        urllib.request.urlretrieve(url, "handel.wav")
    except Exception:
        pass

if os.path.exists("handel.wav"):
    fa, x = wavfile.read("handel.wav"); x = x.astype(float)
    if x.ndim > 1: x = x.mean(1)
    x /= np.max(np.abs(x)); FONTE = "handel.wav"
else:
    # sinal musical SUBSTITUTO: acordes com harmônicos e envelope (fs = 8192 Hz)
    fa = 8192; T = 8.9; t = np.arange(int(T * fa)) / fa
    rng = np.random.default_rng(7)
    acordes = [[261.6, 329.6, 392.0], [293.7, 370.0, 440.0], [329.6, 415.3, 493.9],
               [349.2, 440.0, 523.3], [392.0, 493.9, 587.3], [261.6, 329.6, 392.0]]
    x = np.zeros_like(t); dur = T / len(acordes)
    for i, ac in enumerate(acordes):
        m = (t >= i * dur) & (t < (i + 1) * dur); tt = t[m] - i * dur
        env = (1 - np.exp(-tt / 0.03)) * np.exp(-tt / 1.2)
        for f0 in ac:
            for k in range(1, 7):
                if f0 * k < fa / 2:
                    fase = rng.uniform(0, 6.28)
                    vib = 1 + 0.003 * np.sin(2 * np.pi * 5 * tt)
                    x[m] += env * np.sin(2 * np.pi * f0 * k * tt + fase) * 0.6 ** (k - 1) * vib
    x /= np.max(np.abs(x)) * 1.05; FONTE = "substituto"

t = np.arange(len(x)) / fa
interf = 0.05 * np.cos(200 * np.pi * t) + 0.075 * np.sin(4000 * np.pi * t) # 100 Hz e 2000 Hz
VARS = [1e-2, 1e-1, 1.0] # variâncias do ruído branco
rng = np.random.default_rng(1)
Y = {v: x + interf + rng.normal(0, np.sqrt(v), len(x)) for v in VARS} # y(t) para cada σ²
RES["q5"] = {"fonte": FONTE, "fs": fa, "dur": round(len(x) / fa, 2),
             "Px": float(np.mean(x ** 2))}

def spec(s, fs_):
    """Densidade espectral de potência (Welch) em dB/Hz."""
    f, P = sig.welch(s, fs_, nperseg=2048)
    return f, 10 * np.log10(P + 1e-15)

fig, ax = plt.subplots(2, 1, figsize=(3.4, 4.2))
seg = (t > 2.0) & (t < 2.05)
ax[0].plot(t[seg], Y[1.0][seg], lw=.6, label="y, σ² = 1")
ax[0].plot(t[seg], Y[0.1][seg],"darkorchid", lw=.6, label="y, σ² = 0,1")
ax[0].plot(t[seg], Y[0.01][seg], lw=.8, label="y, σ² = 0,01")
ax[0].plot(t[seg], x[seg], "k", lw=1.3, label="x(t)")
ax[0].set_xlabel("Tempo (s)"); ax[0].set_ylabel("Amplitude"); ax[0].legend(fontsize=5.5, ncol=4)
ax[0].set_title("Domínio do tempo (trecho de 50 ms)")

for v in VARS[::-1]:
    f, P = spec(Y[v], fa); ax[1].plot(f, P, lw=.8, label=f"y, σ² = {v:g}")
f, P = spec(x, fa); ax[1].plot(f, P, "k", lw=1.2, label="x(t)")
for xx in (100, 2000): ax[1].axvline(xx, c="r", ls=":", lw=.8) # interferências
ax[1].set_xlabel("Frequência (Hz)"); ax[1].set_ylabel("DEP (dB/Hz)")
ax[1].set_title("Domínio da frequência (Welch)"); ax[1].legend(fontsize=5.5, ncol=2)

fig.tight_layout()
save(fig, "q5a") # -> figs/q5a.png
display(Image(filename="figs/q5a.png"))

##Código utilizado na questão 5-b

In [ ]:
import os, json, urllib.request
import numpy as np
import scipy.signal as sig
from scipy.io import wavfile
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Image, display

OUT = "figs"
os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({
    "font.size": 6.5, "axes.titlesize": 7, "axes.labelsize": 6.5,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 5.5,
    "xtick.major.size": 2, "ytick.major.size": 2, "lines.linewidth": 0.9,
    "axes.grid": True, "grid.alpha": .35,
    "figure.dpi": 220, "savefig.dpi": 220, "savefig.bbox": "tight"
})
RES = {}

def block(tipo, fc, fs, r):
    th = 2 * np.pi * fc / fs
    a = np.array([1, -2 * r * np.cos(th), r ** 2])
    if tipo == "lp":
        b = np.array([1, 2, 1]); w0 = 0
    elif tipo == "hp":
        b = np.array([1, -2, 1]); w0 = np.pi
    elif tipo == "bp":
        b = np.array([1, 0, -1]); w0 = th
    elif tipo == "notch":
        b = np.array([1, -2 * np.cos(th), 1]); w0 = 0
    _, h = sig.freqz(b, a, worN=[w0])
    return b / abs(h[0]), a

def cascade(blocks):
    b, a = np.array([1.0]), np.array([1.0])
    for bi, ai in blocks:
        b, a = np.convolve(b, bi), np.convolve(a, ai)
    return b, a

def parallel(blocks):
    b, a = np.array([0.0]), np.array([1.0])
    for bi, ai in blocks:
        b = np.polyadd(np.convolve(b, ai), np.convolve(bi, a))
        a = np.convolve(a, ai)
    return b, a

def freq(b, a, fs, n=4096):
    f, h = sig.freqz(b, a, worN=n, fs=fs)
    return f, 20 * np.log10(np.abs(h) + 1e-12)

def pz(ax, b, a, title="", leg=True):
    z, p = np.roots(b), np.roots(a)
    t = np.linspace(0, 2 * np.pi, 400)
    ax.plot(np.cos(t), np.sin(t), "k--", lw=.7)
    ax.plot(z.real, z.imag, "o", mfc="none", mec="b", ms=5, label="zeros")
    ax.plot(p.real, p.imag, "x", c="r", ms=5, label="polos")
    ax.set_aspect("equal"); ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
    ax.set_xlabel("Re{z}"); ax.set_ylabel("Im{z}"); ax.set_title(title)
    if leg: ax.legend(fontsize=5, loc="upper right", framealpha=.6)

def bandwidth3db(b, a, fs):
    f, H = freq(b, a, fs, 20000)
    m = H > H.max() - 3
    return f[m].min(), f[m].max()

def save(fig, name):
    fig.savefig(f"{OUT}/{name}.png"); plt.close(fig)

def quant(c, bits):
    c = np.asarray(c, float)
    ib = max(0, int(np.ceil(np.log2(np.max(np.abs(c)) + 1e-12))))
    fb = bits - 1 - ib
    return np.round(c * 2.0 ** fb) / 2.0 ** fb

def quantb(b, bits):
    b = np.asarray(b, float)
    i = np.flatnonzero(np.abs(b) > 1e-12)[0]
    g = b[i]
    return quant([g], bits)[0] * quant(b / g, bits)

def maxpole(a):
    p = np.roots(a)
    return float(np.max(np.abs(p))) if len(p) else 0.0

if not os.path.exists("handel.wav"):
    try:
        url = "https://github.com/ThayseBachmeyer/ProcessamentoSinaisI/raw/refs/heads/main/Aula%2005/Dados/handel.wav"
        urllib.request.urlretrieve(url, "handel.wav")
    except Exception:
        pass

if os.path.exists("handel.wav"):
    fa, x = wavfile.read("handel.wav"); x = x.astype(float)
    if x.ndim > 1: x = x.mean(1)
    x /= np.max(np.abs(x)); FONTE = "handel.wav"
else:
    # sinal musical SUBSTITUTO: acordes com harmônicos e envelope (fs = 8192 Hz)
    fa = 8192; T = 8.9; t = np.arange(int(T * fa)) / fa
    rng = np.random.default_rng(7)
    acordes = [[261.6, 329.6, 392.0], [293.7, 370.0, 440.0], [329.6, 415.3, 493.9],
               [349.2, 440.0, 523.3], [392.0, 493.9, 587.3], [261.6, 329.6, 392.0]]
    x = np.zeros_like(t); dur = T / len(acordes)
    for i, ac in enumerate(acordes):
        m = (t >= i * dur) & (t < (i + 1) * dur); tt = t[m] - i * dur
        env = (1 - np.exp(-tt / 0.03)) * np.exp(-tt / 1.2)
        for f0 in ac:
            for k in range(1, 7):
                if f0 * k < fa / 2:
                    fase = rng.uniform(0, 6.28)
                    vib = 1 + 0.003 * np.sin(2 * np.pi * 5 * tt)
                    x[m] += env * np.sin(2 * np.pi * f0 * k * tt + fase) * 0.6 ** (k - 1) * vib
    x /= np.max(np.abs(x)) * 1.05; FONTE = "substituto"

t = np.arange(len(x)) / fa
interf = 0.05 * np.cos(200 * np.pi * t) + 0.075 * np.sin(4000 * np.pi * t) # 100 Hz e 2000 Hz
VARS = [1e-2, 1e-1, 1.0] # variâncias do ruído branco
rng = np.random.default_rng(1)
Y = {v: x + interf + rng.normal(0, np.sqrt(v), len(x)) for v in VARS} # y(t) para cada σ²
RES["q5"] = {"fonte": FONTE, "fs": fa, "dur": round(len(x) / fa, 2),
             "Px": float(np.mean(x ** 2))}

def spec(s, fs_):
    """Densidade espectral de potência (Welch) em dB/Hz."""
    f, P = sig.welch(s, fs_, nperseg=2048)
    return f, 10 * np.log10(P + 1e-15)

fig, ax = plt.subplots(2, 1, figsize=(3.4, 4.2))
seg = (t > 2.0) & (t < 2.05)
ax[0].plot(t[seg], Y[1.0][seg], lw=.6, label="y, σ² = 1")
ax[0].plot(t[seg], Y[0.1][seg],"darkorchid", lw=.6, label="y, σ² = 0,1")
ax[0].plot(t[seg], Y[0.01][seg], lw=.8, label="y, σ² = 0,01")
ax[0].plot(t[seg], x[seg], "k", lw=1.3, label="x(t)")
ax[0].set_xlabel("Tempo (s)"); ax[0].set_ylabel("Amplitude"); ax[0].legend(fontsize=5.5, ncol=4)
ax[0].set_title("Domínio do tempo (trecho de 50 ms)")

for v in VARS[::-1]:
    f, P = spec(Y[v], fa); ax[1].plot(f, P, lw=.8, label=f"y, σ² = {v:g}")
f, P = spec(x, fa); ax[1].plot(f, P, "k", lw=1.2, label="x(t)")
for xx in (100, 2000): ax[1].axvline(xx, c="r", ls=":", lw=.8) # interferências
ax[1].set_xlabel("Frequência (Hz)"); ax[1].set_ylabel("DEP (dB/Hz)")
ax[1].set_title("Domínio da frequência (Welch)"); ax[1].legend(fontsize=5.5, ncol=2)



fig, ax = plt.subplots(1, 2, figsize=(3.4, 2.1), sharey=True)
for i, (s, tt_) in enumerate([(x, "x(t)"), (Y[0.01], "y(t), σ² = 0,01")]):
    ax[i].specgram(s, NFFT=512, Fs=fa, noverlap=384, cmap="magma", vmin=-110)
    ax[i].set_title(tt_); ax[i].set_xlabel("Tempo (s)"); ax[i].grid(False)
ax[0].set_ylabel("Frequência (Hz)")
fig.tight_layout();
save(fig, "q5b") # -> q5b.png
fs_, P = spec(x, fa) # % da energia de x acima de 3 kHz
RES["q5"]["energia_acima_3k_pct"] = float(100 * np.sum(10 ** (P[fs_ > 3000] / 10)) / np.sum(10 ** (P / 10)))

display(Image(filename="figs/q5b.png"))

##Código utilizado na questão 5-c

In [ ]:
import os, json, urllib.request
import numpy as np
import scipy.signal as sig
from scipy.io import wavfile
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Image, display

OUT = "figs"
os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({
    "font.size": 6.5, "axes.titlesize": 7, "axes.labelsize": 6.5,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 5.5,
    "xtick.major.size": 2, "ytick.major.size": 2, "lines.linewidth": 0.9,
    "axes.grid": True, "grid.alpha": .35,
    "figure.dpi": 220, "savefig.dpi": 220, "savefig.bbox": "tight"
})
RES = {}

def block(tipo, fc, fs, r):
    th = 2 * np.pi * fc / fs
    a = np.array([1, -2 * r * np.cos(th), r ** 2])
    if tipo == "lp":
        b = np.array([1, 2, 1]); w0 = 0
    elif tipo == "hp":
        b = np.array([1, -2, 1]); w0 = np.pi
    elif tipo == "bp":
        b = np.array([1, 0, -1]); w0 = th
    elif tipo == "notch":
        b = np.array([1, -2 * np.cos(th), 1]); w0 = 0
    _, h = sig.freqz(b, a, worN=[w0])
    return b / abs(h[0]), a

def cascade(blocks):
    b, a = np.array([1.0]), np.array([1.0])
    for bi, ai in blocks:
        b, a = np.convolve(b, bi), np.convolve(a, ai)
    return b, a

def parallel(blocks):
    b, a = np.array([0.0]), np.array([1.0])
    for bi, ai in blocks:
        b = np.polyadd(np.convolve(b, ai), np.convolve(bi, a))
        a = np.convolve(a, ai)
    return b, a

def freq(b, a, fs, n=4096):
    f, h = sig.freqz(b, a, worN=n, fs=fs)
    return f, 20 * np.log10(np.abs(h) + 1e-12)

def pz(ax, b, a, title="", leg=True):
    z, p = np.roots(b), np.roots(a)
    t = np.linspace(0, 2 * np.pi, 400)
    ax.plot(np.cos(t), np.sin(t), "k--", lw=.7)
    ax.plot(z.real, z.imag, "o", mfc="none", mec="b", ms=5, label="zeros")
    ax.plot(p.real, p.imag, "x", c="r", ms=5, label="polos")
    ax.set_aspect("equal"); ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
    ax.set_xlabel("Re{z}"); ax.set_ylabel("Im{z}"); ax.set_title(title)
    if leg: ax.legend(fontsize=5, loc="upper right", framealpha=.6)

def bandwidth3db(b, a, fs):
    f, H = freq(b, a, fs, 20000)
    m = H > H.max() - 3
    return f[m].min(), f[m].max()

def save(fig, name):
    fig.savefig(f"{OUT}/{name}.png"); plt.close(fig)

def quant(c, bits):
    c = np.asarray(c, float)
    ib = max(0, int(np.ceil(np.log2(np.max(np.abs(c)) + 1e-12))))
    fb = bits - 1 - ib
    return np.round(c * 2.0 ** fb) / 2.0 ** fb

def quantb(b, bits):
    b = np.asarray(b, float)
    i = np.flatnonzero(np.abs(b) > 1e-12)[0]
    g = b[i]
    return quant([g], bits)[0] * quant(b / g, bits)

def maxpole(a):
    p = np.roots(a)
    return float(np.max(np.abs(p))) if len(p) else 0.0

if not os.path.exists("handel.wav"):
    try:
        url = "https://github.com/ThayseBachmeyer/ProcessamentoSinaisI/raw/refs/heads/main/Aula%2005/Dados/handel.wav"
        urllib.request.urlretrieve(url, "handel.wav")
    except Exception:
        pass

if os.path.exists("handel.wav"):
    fa, x = wavfile.read("handel.wav"); x = x.astype(float)
    if x.ndim > 1: x = x.mean(1)
    x /= np.max(np.abs(x)); FONTE = "handel.wav"
else:
    # sinal musical SUBSTITUTO: acordes com harmônicos e envelope (fs = 8192 Hz)
    fa = 8192; T = 8.9; t = np.arange(int(T * fa)) / fa
    rng = np.random.default_rng(7)
    acordes = [[261.6, 329.6, 392.0], [293.7, 370.0, 440.0], [329.6, 415.3, 493.9],
               [349.2, 440.0, 523.3], [392.0, 493.9, 587.3], [261.6, 329.6, 392.0]]
    x = np.zeros_like(t); dur = T / len(acordes)
    for i, ac in enumerate(acordes):
        m = (t >= i * dur) & (t < (i + 1) * dur); tt = t[m] - i * dur
        env = (1 - np.exp(-tt / 0.03)) * np.exp(-tt / 1.2)
        for f0 in ac:
            for k in range(1, 7):
                if f0 * k < fa / 2:
                    fase = rng.uniform(0, 6.28)
                    vib = 1 + 0.003 * np.sin(2 * np.pi * 5 * tt)
                    x[m] += env * np.sin(2 * np.pi * f0 * k * tt + fase) * 0.6 ** (k - 1) * vib
    x /= np.max(np.abs(x)) * 1.05; FONTE = "substituto"

t = np.arange(len(x)) / fa
interf = 0.05 * np.cos(200 * np.pi * t) + 0.075 * np.sin(4000 * np.pi * t) # 100 Hz e 2000 Hz
VARS = [1e-2, 1e-1, 1.0] # variâncias do ruído branco
rng = np.random.default_rng(1)
Y = {v: x + interf + rng.normal(0, np.sqrt(v), len(x)) for v in VARS} # y(t) para cada σ²
RES["q5"] = {"fonte": FONTE, "fs": fa, "dur": round(len(x) / fa, 2),
             "Px": float(np.mean(x ** 2))}

def spec(s, fs_):
    """Densidade espectral de potência (Welch) em dB/Hz."""
    f, P = sig.welch(s, fs_, nperseg=2048)
    return f, 10 * np.log10(P + 1e-15)

fig, ax = plt.subplots(2, 1, figsize=(3.4, 4.2))
seg = (t > 2.0) & (t < 2.05)
ax[0].plot(t[seg], Y[1.0][seg], lw=.6, label="y, σ² = 1")
ax[0].plot(t[seg], Y[0.1][seg], "darkorchid", lw=.6, label="y, σ² = 0,1")
ax[0].plot(t[seg], Y[0.01][seg], lw=.8, label="y, σ² = 0,01")
ax[0].plot(t[seg], x[seg], "k", lw=1.3, label="x(t)")
ax[0].set_xlabel("Tempo (s)"); ax[0].set_ylabel("Amplitude"); ax[0].legend(fontsize=5.5, ncol=4)
ax[0].set_title("Domínio do tempo (trecho de 50 ms)")

for v in VARS[::-1]:
    f, P = spec(Y[v], fa); ax[1].plot(f, P, lw=.8, label=f"y, σ² = {v:g}")
f, P = spec(x, fa); ax[1].plot(f, P, "k", lw=1.2, label="x(t)")
for xx in (100, 2000): ax[1].axvline(xx, c="r", ls=":", lw=.8) # interferências
ax[1].set_xlabel("Frequência (Hz)"); ax[1].set_ylabel("DEP (dB/Hz)")
ax[1].set_title("Domínio da frequência (Welch)"); ax[1].legend(fontsize=5.5, ncol=2)
plt.close(fig)



FC_LP = 3200
SIST = [("Notch 100 Hz", block("notch", 100, fa, 0.99)), # remove interferência de 100 Hz
        ("Notch 2000 Hz", block("notch", 2000, fa, 0.99)), # remove tom de 2 kHz
        ("Passa-baixas 3,2 kHz (1)", block("lp", FC_LP, fa, 0.55)), # corta ruído acima da banda útil
        ("Passa-baixas 3,2 kHz (2)", block("lp", FC_LP, fa, 0.55))]
RES["q5"]["filtros"] = [{"nome": n, "b": b.round(6).tolist(), "a": a.round(6).tolist()}
                        for n, (b, a) in SIST]
fig, ax = plt.subplots(3, 2, figsize=(3.4, 4.8), gridspec_kw={"width_ratios": [1.5, 1]})
for i, (n, (b, a)) in enumerate(SIST[:3]): # blocos distintos (o 4º é igual ao 3º)
    f, H = freq(b, a, fa, 8192); ax[i, 0].plot(f, H); ax[i, 0].set_title(n)
    ax[i, 0].set_ylim(-60, 5); ax[i, 0].set_xlabel("Frequência (Hz)"); ax[i, 0].set_ylabel("|H| (dB)")
    pz(ax[i, 1], b, a, "Polos (x) e zeros (o)", leg=False)
fig.tight_layout(); save(fig, "q5c_blocos") # -> q5c_blocos.png

bS, aS = cascade([ba for _, ba in SIST]) # sistema completo (8ª ordem)
fig, ax = plt.subplots(2, 1, figsize=(3.4, 5.0), gridspec_kw={"height_ratios": [1, 1.2]})
f, H = freq(bS, aS, fa, 8192); ax[0].plot(f, H, "k")
for xx in (100, 2000, FC_LP): ax[0].axvline(xx, c="r", ls=":", lw=.8)
ax[0].set_ylim(-80, 5); ax[0].set_xlabel("Frequência (Hz)"); ax[0].set_ylabel("|H| (dB)")
ax[0].set_title("Sistema completo (4 blocos em cascata)")
pz(ax[1], bS, aS, "Polos e zeros do sistema")
fig.tight_layout(); save(fig, "q5c_sistema") # -> q5c_sistema.png

SOS = np.array([np.concatenate([b, a]) for _, (b, a) in SIST]) # matriz de seções de 2ª ordem

def recupera(y, sos=SOS):
    """Aplica o sistema em cascata (seções de 2ª ordem)."""
    return sig.sosfilt(sos, y)

display(Image(filename="figs/q5c_blocos.png"))
display(Image(filename="figs/q5c_sistema.png"))

##Código utilizado na questão 5-d / 5-e / 5-f

In [ ]:
import os, json, urllib.request
import numpy as np
import scipy.signal as sig
from scipy.io import wavfile
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Image, Audio, display

OUT = "figs"
os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({
    "font.size": 6.5, "axes.titlesize": 7, "axes.labelsize": 6.5,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 5.5,
    "xtick.major.size": 2, "ytick.major.size": 2, "lines.linewidth": 0.9,
    "axes.grid": True, "grid.alpha": .35,
    "figure.dpi": 220, "savefig.dpi": 220, "savefig.bbox": "tight"
})
RES = {}

def block(tipo, fc, fs, r):
    th = 2 * np.pi * fc / fs
    a = np.array([1, -2 * r * np.cos(th), r ** 2])
    if tipo == "lp":
        b = np.array([1, 2, 1]); w0 = 0
    elif tipo == "hp":
        b = np.array([1, -2, 1]); w0 = np.pi
    elif tipo == "bp":
        b = np.array([1, 0, -1]); w0 = th
    elif tipo == "notch":
        b = np.array([1, -2 * np.cos(th), 1]); w0 = 0
    _, h = sig.freqz(b, a, worN=[w0])
    return b / abs(h[0]), a

def cascade(blocks):
    b, a = np.array([1.0]), np.array([1.0])
    for bi, ai in blocks:
        b, a = np.convolve(b, bi), np.convolve(a, ai)
    return b, a

def parallel(blocks):
    b, a = np.array([0.0]), np.array([1.0])
    for bi, ai in blocks:
        b = np.polyadd(np.convolve(b, ai), np.convolve(bi, a))
        a = np.convolve(a, ai)
    return b, a

def freq(b, a, fs, n=4096):
    f, h = sig.freqz(b, a, worN=n, fs=fs)
    return f, 20 * np.log10(np.abs(h) + 1e-12)

def pz(ax, b, a, title="", leg=True):
    z, p = np.roots(b), np.roots(a)
    t = np.linspace(0, 2 * np.pi, 400)
    ax.plot(np.cos(t), np.sin(t), "k--", lw=.7)
    ax.plot(z.real, z.imag, "o", mfc="none", mec="b", ms=5, label="zeros")
    ax.plot(p.real, p.imag, "x", c="r", ms=5, label="polos")
    ax.set_aspect("equal"); ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
    ax.set_xlabel("Re{z}"); ax.set_ylabel("Im{z}"); ax.set_title(title)
    if leg: ax.legend(fontsize=5, loc="upper right", framealpha=.6)

def bandwidth3db(b, a, fs):
    f, H = freq(b, a, fs, 20000)
    m = H > H.max() - 3
    return f[m].min(), f[m].max()

def save(fig, name):
    fig.savefig(f"{OUT}/{name}.png"); plt.close(fig)

def quant(c, bits):
    c = np.asarray(c, float)
    ib = max(0, int(np.ceil(np.log2(np.max(np.abs(c)) + 1e-12))))
    fb = bits - 1 - ib
    return np.round(c * 2.0 ** fb) / 2.0 ** fb

def quantb(b, bits):
    b = np.asarray(b, float)
    i = np.flatnonzero(np.abs(b) > 1e-12)[0]
    g = b[i]
    return quant([g], bits)[0] * quant(b / g, bits)

def maxpole(a):
    p = np.roots(a)
    return float(np.max(np.abs(p))) if len(p) else 0.0

if not os.path.exists("handel.wav"):
    try:
        url = "https://github.com/ThayseBachmeyer/ProcessamentoSinaisI/raw/refs/heads/main/Aula%2005/Dados/handel.wav"
        urllib.request.urlretrieve(url, "handel.wav")
    except Exception:
        pass

if os.path.exists("handel.wav"):
    fa, x = wavfile.read("handel.wav"); x = x.astype(float)
    if x.ndim > 1: x = x.mean(1)
    x /= np.max(np.abs(x)); FONTE = "handel.wav"
else:
    # sinal musical SUBSTITUTO: acordes com harmônicos e envelope (fs = 8192 Hz)
    fa = 8192; T = 8.9; t = np.arange(int(T * fa)) / fa
    rng = np.random.default_rng(7)
    acordes = [[261.6, 329.6, 392.0], [293.7, 370.0, 440.0], [329.6, 415.3, 493.9],
               [349.2, 440.0, 523.3], [392.0, 493.9, 587.3], [261.6, 329.6, 392.0]]
    x = np.zeros_like(t); dur = T / len(acordes)
    for i, ac in enumerate(acordes):
        m = (t >= i * dur) & (t < (i + 1) * dur); tt = t[m] - i * dur
        env = (1 - np.exp(-tt / 0.03)) * np.exp(-tt / 1.2)
        for f0 in ac:
            for k in range(1, 7):
                if f0 * k < fa / 2:
                    fase = rng.uniform(0, 6.28)
                    vib = 1 + 0.003 * np.sin(2 * np.pi * 5 * tt)
                    x[m] += env * np.sin(2 * np.pi * f0 * k * tt + fase) * 0.6 ** (k - 1) * vib
    x /= np.max(np.abs(x)) * 1.05; FONTE = "substituto"

t = np.arange(len(x)) / fa
interf = 0.05 * np.cos(200 * np.pi * t) + 0.075 * np.sin(4000 * np.pi * t) # 100 Hz e 2000 Hz
VARS = [1e-2, 1e-1, 1.0] # variâncias do ruído branco
rng = np.random.default_rng(1)
Y = {v: x + interf + rng.normal(0, np.sqrt(v), len(x)) for v in VARS} # y(t) para cada σ²
RES["q5"] = {"fonte": FONTE, "fs": fa, "dur": round(len(x) / fa, 2),
             "Px": float(np.mean(x ** 2))}

def spec(s, fs_):
    """Densidade espectral de potência (Welch) em dB/Hz."""
    f, P = sig.welch(s, fs_, nperseg=2048)
    return f, 10 * np.log10(P + 1e-15)

FC_LP = 3200
SIST = [("Notch 100 Hz", block("notch", 100, fa, 0.99)),
        ("Notch 2000 Hz", block("notch", 2000, fa, 0.99)),
        ("Passa-baixas 3,2 kHz (1)", block("lp", FC_LP, fa, 0.55)),
        ("Passa-baixas 3,2 kHz (2)", block("lp", FC_LP, fa, 0.55))]
SOS = np.array([np.concatenate([b, a]) for _, (b, a) in SIST])

def recupera(y, sos=SOS):
    """Aplica o sistema em cascata (seções de 2ª ordem)."""
    return sig.sosfilt(sos, y)

XR = recupera(x) # referência justa: x passando pelo mesmo sistema (mesma fase/atraso)

def metricas(xref, xh):
    e = xh - xref
    mse = float(np.mean(e ** 2))
    return {"MSE": mse, "SNR": float(10 * np.log10(np.mean(xref ** 2) / mse)), "Pres": mse}

RES["q5"]["metricas"] = {}
REC = {}
for v in VARS:
    y = Y[v]; xh = recupera(y); REC[v] = xh
    m_in = metricas(x, y) # antes da filtragem
    m_out = metricas(XR, xh) # depois: erro = ruído/interferência residual
    m_abs = metricas(x, xh) # erro total vs x original (inclui distorção de fase/banda)
    RES["q5"]["metricas"][str(v)] = {"SNR_in": m_in["SNR"], "MSE_in": m_in["MSE"],
                                     "SNR_out": m_out["SNR"], "MSE_out": m_out["MSE"], "Pres": m_out["Pres"],
                                     "SNR_abs": m_abs["SNR"], "MSE_abs": m_abs["MSE"]}
    if v == 0.01: # (f) áudios para ouvir
        wavfile.write("audio_y_0p01.wav", fa, (y / np.max(np.abs(y)) * 0.9).astype(np.float32))
        wavfile.write("audio_xhat_0p01.wav", fa, (xh / np.max(np.abs(xh)) * 0.9).astype(np.float32))

fig, ax = plt.subplots(2, 1, figsize=(3.4, 4.2))
seg = (t > 2.0) & (t < 2.04)
ax[0].plot(t[seg], Y[0.01][seg], c="tab:orange", lw=.7, label="y(t)")
ax[0].plot(t[seg], x[seg], "k", lw=1.3, label="x(t)")
ax[0].plot(t[seg], REC[0.01][seg], c="tab:blue", lw=1, ls="--", label="x̂(t)")
ax[0].set_title("Tempo (σ² = 0,01, trecho de 40 ms)"); ax[0].set_xlabel("Tempo (s)")
ax[0].legend(fontsize=5.5, ncol=3)
for s, l, c in [(Y[0.01], "y(t)", "tab:orange"), (x, "x(t)", "k"), (REC[0.01], "x̂(t)", "tab:blue")]:
    f, P = spec(s, fa); ax[1].plot(f, P, c=c, lw=1, label=l)
ax[1].set_title("Frequência (σ² = 0,01)")
ax[1].set_xlabel("Frequência (Hz)"); ax[1].set_ylabel("DEP (dB/Hz)")
ax[1].legend(fontsize=5.5)
fig.tight_layout(); save(fig, "q5d_001") # -> q5d_001.png

fig, ax = plt.subplots(1, 2, figsize=(3.4, 2.1), sharey=True)
for i, v in enumerate([0.1, 1.0]):
    for s, l, c in [(Y[v], "y(t)", "tab:orange"), (x, "x(t)", "k"), (REC[v], "x̂(t)", "tab:blue")]:
        f, P = spec(s, fa); ax[i].plot(f, P, c=c, lw=1, label=l)
    ax[i].set_title(f"σ² = {v:g}"); ax[i].set_xlabel("Frequência (Hz)")
ax[0].set_ylabel("DEP (dB/Hz)"); ax[0].legend(fontsize=5.5)
fig.tight_layout(); save(fig, "q5d_outros") # -> q5d_outros.png

display(Image(filename="figs/q5d_001.png"))
display(Image(filename="figs/q5d_outros.png"))
print(json.dumps(RES["q5"]["metricas"], indent=2))

print("Áudio Original x(t):")
display(Audio(x, rate=fa))

print("Áudio Corrompido y(t) (σ² = 0,01):")
display(Audio("audio_y_0p01.wav"))

print("Áudio Recuperado x̂(t) (σ² = 0,01):")
display(Audio("audio_xhat_0p01.wav"))

##Código utilizado na questão 5-g

In [ ]:
import os, json, urllib.request
import numpy as np
import scipy.signal as sig
from scipy.io import wavfile
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Image, display

OUT = "figs"
os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({
    "font.size": 6.5, "axes.titlesize": 7, "axes.labelsize": 6.5,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 5.5,
    "xtick.major.size": 2, "ytick.major.size": 2, "lines.linewidth": 0.9,
    "axes.grid": True, "grid.alpha": .35,
    "figure.dpi": 220, "savefig.dpi": 220, "savefig.bbox": "tight"
})
RES = {}

def block(tipo, fc, fs, r):
    th = 2 * np.pi * fc / fs
    a = np.array([1, -2 * r * np.cos(th), r ** 2])
    if tipo == "lp":
        b = np.array([1, 2, 1]); w0 = 0
    elif tipo == "hp":
        b = np.array([1, -2, 1]); w0 = np.pi
    elif tipo == "bp":
        b = np.array([1, 0, -1]); w0 = th
    elif tipo == "notch":
        b = np.array([1, -2 * np.cos(th), 1]); w0 = 0
    _, h = sig.freqz(b, a, worN=[w0])
    return b / abs(h[0]), a

def cascade(blocks):
    b, a = np.array([1.0]), np.array([1.0])
    for bi, ai in blocks:
        b, a = np.convolve(b, bi), np.convolve(a, ai)
    return b, a

def parallel(blocks):
    b, a = np.array([0.0]), np.array([1.0])
    for bi, ai in blocks:
        b = np.polyadd(np.convolve(b, ai), np.convolve(bi, a))
        a = np.convolve(a, ai)
    return b, a

def freq(b, a, fs, n=4096):
    f, h = sig.freqz(b, a, worN=n, fs=fs)
    return f, 20 * np.log10(np.abs(h) + 1e-12)

def pz(ax, b, a, title="", leg=True):
    z, p = np.roots(b), np.roots(a)
    t = np.linspace(0, 2 * np.pi, 400)
    ax.plot(np.cos(t), np.sin(t), "k--", lw=.7)
    ax.plot(z.real, z.imag, "o", mfc="none", mec="b", ms=5, label="zeros")
    ax.plot(p.real, p.imag, "x", c="r", ms=5, label="polos")
    ax.set_aspect("equal"); ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
    ax.set_xlabel("Re{z}"); ax.set_ylabel("Im{z}"); ax.set_title(title)
    if leg: ax.legend(fontsize=5, loc="upper right", framealpha=.6)

def bandwidth3db(b, a, fs):
    f, H = freq(b, a, fs, 20000)
    m = H > H.max() - 3
    return f[m].min(), f[m].max()

def save(fig, name):
    fig.savefig(f"{OUT}/{name}.png"); plt.close(fig)

def quant(c, bits):
    c = np.asarray(c, float)
    ib = max(0, int(np.ceil(np.log2(np.max(np.abs(c)) + 1e-12))))
    fb = bits - 1 - ib
    return np.round(c * 2.0 ** fb) / 2.0 ** fb

def quantb(b, bits):
    b = np.asarray(b, float)
    i = np.flatnonzero(np.abs(b) > 1e-12)[0]
    g = b[i]
    return quant([g], bits)[0] * quant(b / g, bits)

def maxpole(a):
    p = np.roots(a)
    return float(np.max(np.abs(p))) if len(p) else 0.0

if not os.path.exists("handel.wav"):
    try:
        url = "https://github.com/ThayseBachmeyer/ProcessamentoSinaisI/raw/refs/heads/main/Aula%2005/Dados/handel.wav"
        urllib.request.urlretrieve(url, "handel.wav")
    except Exception:
        pass

if os.path.exists("handel.wav"):
    fa, x = wavfile.read("handel.wav"); x = x.astype(float)
    if x.ndim > 1: x = x.mean(1)
    x /= np.max(np.abs(x)); FONTE = "handel.wav"
else:
    # sinal musical SUBSTITUTO: acordes com harmônicos e envelope (fs = 8192 Hz)
    fa = 8192; T = 8.9; t = np.arange(int(T * fa)) / fa
    rng = np.random.default_rng(7)
    acordes = [[261.6, 329.6, 392.0], [293.7, 370.0, 440.0], [329.6, 415.3, 493.9],
               [349.2, 440.0, 523.3], [392.0, 493.9, 587.3], [261.6, 329.6, 392.0]]
    x = np.zeros_like(t); dur = T / len(acordes)
    for i, ac in enumerate(acordes):
        m = (t >= i * dur) & (t < (i + 1) * dur); tt = t[m] - i * dur
        env = (1 - np.exp(-tt / 0.03)) * np.exp(-tt / 1.2)
        for f0 in ac:
            for k in range(1, 7):
                if f0 * k < fa / 2:
                    fase = rng.uniform(0, 6.28)
                    vib = 1 + 0.003 * np.sin(2 * np.pi * 5 * tt)
                    x[m] += env * np.sin(2 * np.pi * f0 * k * tt + fase) * 0.6 ** (k - 1) * vib
    x /= np.max(np.abs(x)) * 1.05; FONTE = "substituto"

t = np.arange(len(x)) / fa
interf = 0.05 * np.cos(200 * np.pi * t) + 0.075 * np.sin(4000 * np.pi * t) # 100 Hz e 2000 Hz
VARS = [1e-2, 1e-1, 1.0] # variâncias do ruído branco
rng = np.random.default_rng(1)
Y = {v: x + interf + rng.normal(0, np.sqrt(v), len(x)) for v in VARS} # y(t) para cada σ²
RES["q5"] = {"fonte": FONTE, "fs": fa, "dur": round(len(x) / fa, 2),
             "Px": float(np.mean(x ** 2))}

FC_LP = 3200
SIST = [("Notch 100 Hz", block("notch", 100, fa, 0.99)),
        ("Notch 2000 Hz", block("notch", 2000, fa, 0.99)),
        ("Passa-baixas 3,2 kHz (1)", block("lp", FC_LP, fa, 0.55)),
        ("Passa-baixas 3,2 kHz (2)", block("lp", FC_LP, fa, 0.55))]
bS, aS = cascade([ba for _, ba in SIST])
SOS = np.array([np.concatenate([b, a]) for _, (b, a) in SIST])
XR = sig.sosfilt(SOS, x)

def metricas(xref, xh):
    e = xh - xref
    mse = float(np.mean(e ** 2))
    return {"MSE": mse, "SNR": float(10 * np.log10(np.mean(xref ** 2) / mse)), "Pres": mse}

BITS5 = [2, 4, 8, 16]
RES["q5"]["quant"] = {}
fig, ax = plt.subplots(2, 2, figsize=(3.5, 3.9))
f, H = freq(bS, aS, fa, 8192)
ax[0, 0].plot(f, H, "k", lw=2, label="original"); ax[0, 1].plot(f, H, "k", lw=2, label="original")
for bits in BITS5:
    sosq = np.array([np.concatenate([quantb(b, bits), quant(a, bits)]) for _, (b, a) in SIST]) # por bloco
    bq, aq = quantb(bS, bits), quant(aS, bits) # forma direta
    bc, ac = cascade([(s[:3], s[3:]) for s in sosq])
    mp_b, mp_d = maxpole(ac), maxpole(aq)
    rec = {}
    if mp_b < 1: # só filtra se estável
        xh = sig.sosfilt(sosq, Y[0.01])
        rec["SNR_blocos"] = metricas(XR, xh)["SNR"]
        fq, Hq = freq(bc, ac, fa, 8192); ax[0, 1].plot(fq, Hq, lw=.9, label=f"{bits} bits")
    if mp_d < 1 and not np.allclose(bq, 0):
        xh = sig.lfilter(bq, aq, Y[0.01])
        rec["SNR_direta"] = metricas(XR, xh)["SNR"] if np.all(np.isfinite(xh)) else None
        fq, Hq = freq(bq, aq, fa, 8192); ax[0, 0].plot(fq, Hq, lw=.9, label=f"{bits} bits")
    RES["q5"]["quant"][bits] = {"maxpole_blocos": round(mp_b, 4), "maxpole_direta": round(mp_d, 4), **rec}
    if bits == 8:
        pz(ax[1, 0], bq, aq, "Direta, 8 bits"); pz(ax[1, 1], bc, ac, "Cascata, 8 bits")
for col, tt_ in enumerate(["Forma direta (8ª ordem)", "Cascata de biquads"]):
    ax[0, col].set_ylim(-90, 20); ax[0, col].set_title(tt_)
    ax[0, col].set_xlabel("Frequência (Hz)"); ax[0, col].set_ylabel("|H| (dB)")
    ax[0, col].legend(fontsize=5)
fig.tight_layout(); save(fig, "q5g") # -> figs/q5g.png

display(Image(filename="figs/q5g.png"))
print(json.dumps(RES["q5"]["quant"], indent=2))